# BASIC PROJECT TO GET E

## 1.1

### 1.1.1 *Download a pre-trained modern ConvNet such as: ResNet18, ResNet34,...*

In [4]:
import torchvision.models as models

# download pretrained resnet34
# **THIS ONLY DOWNLOADS THE FIRST TIME IT RUNS, THEN GETS CACHED
model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)

# validate model
print(model)

ResNet(
  (conv1): Conv2d(3, 64, kernel_size=(7, 7), stride=(2, 2), padding=(3, 3), bias=False)
  (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
  (relu): ReLU(inplace=True)
  (maxpool): MaxPool2d(kernel_size=3, stride=2, padding=1, dilation=1, ceil_mode=False)
  (layer1): Sequential(
    (0): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
      (conv2): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn2): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
    )
    (1): BasicBlock(
      (conv1): Conv2d(64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False)
      (bn1): BatchNorm2d(64, eps=1e-05, momentum=0.1, affine=True, track_running_stats=True)
      (relu): ReLU(inplace=True)
  

### 1.1.2 *Download the dataset*

Heads up the file is pretty big so I wasn't able to add it to the git repo, but theres directions in the assignment for where to get the data, and I documented how this code expects your local directory to look in the readme.

In [2]:
# extract data
import tarfile

for fname in ['images.tar.gz', 'annotations.tar.gz']:
    with tarfile.open(f'./dataset/oxford-iiit-pet/{fname}') as tar:
        tar.extractall('./dataset/oxford-iiit-pet/')

/var/folders/jp/90g4mw8n1xl74z8kt8jj8s700000gn/T/ipykernel_11002/2871605484.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall('./dataset/oxford-iiit-pet/')


### 1.1.3 *sanity check*

#### load in data

In [5]:
import base_project as bp

from torch.utils.data import DataLoader
from torchvision import datasets, transforms

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    # normalization values for ImageNet (ResNet)
    transforms.Normalize(
        mean = [0.485, 0.456, 0.406],
        std  = [0.229, 0.224, 0.225]
    )
])

train_data = datasets.OxfordIIITPet(root='./dataset', split='trainval',
                                     download=False, transform=transform)
test_data  = datasets.OxfordIIITPet(root='./dataset', split='test',
                                     download=False, transform=transform)

# convert labels to binary (dog=0, cat=1)
train_dataset = bp.BinaryPetDataset(train_data)
test_dataset = bp.BinaryPetDataset(test_data)

# set dataloader wrappers
train_loader = DataLoader(train_dataset,
                          batch_size=32, 
                          shuffle=True,
                          num_workers=2)
test_loader = DataLoader(test_dataset,
                          batch_size=32, 
                          shuffle=False, 
                          num_workers=2)

In [6]:
import torch
import torch.nn as nn

device = torch.device("mps" if torch.backends.mps.is_available() else "cpu")

# freeze all layers (for sanity check)
for param in model.parameters():
    param.requires_grad = False

# unfreeze layer 4 and fc 
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

# modify network for binary classification
model.fc = nn.Linear(512, 1)
model = model.to(device)

# init optimizer and loss function
optimizer = torch.optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4)
criterion = nn.BCEWithLogitsLoss()

#### actually run sanity check

In [ ]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_accuracy = bp.train_epoch_binary(model, train_loader, optimizer, criterion, device)
    test_loss, test_accuracy = bp.train_epoch_binary(model, test_loader, optimizer, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train loss: {train_loss:.4f}, accuracy: {train_accuracy:.4f} | "
          f"Test loss: {test_loss:.4f}, accuracy: {test_accuracy:.4f}")


Epoch 1/5 | Train loss: 0.0624, accuracy: 0.9802 | Test loss: 0.6892, accuracy: 0.6743
Epoch 2/5 | Train loss: 0.0525, accuracy: 0.9935 | Test loss: 0.3453, accuracy: 0.8915
Epoch 3/5 | Train loss: 0.0374, accuracy: 0.9976 | Test loss: 0.1660, accuracy: 0.9586
Epoch 4/5 | Train loss: 0.0230, accuracy: 0.9978 | Test loss: 0.0743, accuracy: 0.9932
Epoch 5/5 | Train loss: 0.0093, accuracy: 0.9997 | Test loss: 0.0367, accuracy: 0.9995


### 1.1.4 *multi-class linear probing*

In [ ]:
train_loader = DataLoader(train_data,
                          batch_size=32, 
                          shuffle=True,
                          num_workers=2)
test_loader = DataLoader(test_data,
                          batch_size=32, 
                          shuffle=False, 
                          num_workers=2)


model = models.resnet34(weights=models.ResNet34_Weights.DEFAULT)

# freeze model
for param in model.parameters():
    param.requires_grad = False

# only train on new head (linear probing)
model.fc = nn.Linear(512, 37)
model = model.to(device)

optimizer = torch.optim.Adam(model.fc.parameters(), lr=1e-3)
# change loss function to standard cross entropy loss for multi-class
criterion = nn.CrossEntropyLoss()

In [8]:
num_epochs = 5

for epoch in range(num_epochs):
    train_loss, train_acc = bp.train_epoch(model, train_loader, optimizer, criterion, device)
    test_loss,  test_acc  = bp.evaluate_epoch(model, test_loader, criterion, device)

    print(f"Epoch {epoch+1}/{num_epochs} | "
          f"Train loss: {train_loss:.4f}, acc: {train_acc:.4f} | "
          f"Test loss: {test_loss:.4f}, acc: {test_acc:.4f}")

Epoch 1/5 | Train loss: 1.8210, acc: 0.6041 | Test loss: 0.8256, acc: 0.8179
Epoch 2/5 | Train loss: 0.6134, acc: 0.8734 | Test loss: 0.5291, acc: 0.8659
Epoch 3/5 | Train loss: 0.4246, acc: 0.9016 | Test loss: 0.4502, acc: 0.8708
Epoch 4/5 | Train loss: 0.3380, acc: 0.9242 | Test loss: 0.4102, acc: 0.8823
Epoch 5/5 | Train loss: 0.2721, acc: 0.9391 | Test loss: 0.3903, acc: 0.8793
